# 01 — 癌症數據初步探索

**目標**: 探索公開癌症基因體數據集的基本結構、分佈與品質
**數據來源**: TCGA (via UCSC Xena)、cBioPortal
**分析重點**: 基因表現分佈、樣本族群統計、PCA 降維視覺化

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

print("套件載入完成。")

---
## 1. cBioPortal 可用研究清單

In [ ]:
CBIOPORTAL_API = "https://www.cbioportal.org/api"

resp = requests.get(f"{CBIOPORTAL_API}/studies", timeout=30)
studies = resp.json()
print(f"cBioPortal 共有 {len(studies)} 個研究\n")

studies_df = pd.DataFrame(studies)
if not studies_df.empty:
    display_cols = [c for c in ["studyId", "name", "cancerTypeId", "description"] if c in studies_df.columns]
    display(studies_df[display_cols].head(10))

### 各癌症類型研究數量

In [ ]:
if not studies_df.empty and "cancerTypeId" in studies_df.columns:
    cancer_counts = studies_df["cancerTypeId"].value_counts()
    plt.figure(figsize=(14, 6))
    cancer_counts.head(20).plot(kind="bar", color="steelblue", edgecolor="none")
    plt.title("cBioPortal 各癌症類型研究數量 (Top 20)", fontsize=16, fontweight="bold")
    plt.xlabel("癌症類型")
    plt.ylabel("研究數量")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

---
## 2. 查詢特定癌症研究的分子輪廓

In [ ]:
STUDY_ID = "brca_tcga_pan_can_atlas_2018"

resp = requests.get(f"{CBIOPORTAL_API}/studies/{STUDY_ID}/molecular-profiles", timeout=30)
profiles = resp.json()
print(f"\"{STUDY_ID}\" 共有 {len(profiles)} 個分子輪廓\n")

profiles_df = pd.DataFrame(profiles)
if not profiles_df.empty:
    cols = [c for c in ["molecularAlterationType", "datatype", "name", "description"] if c in profiles_df.columns]
    display(profiles_df[cols])

---
## 3. 下載基因表現數據

In [ ]:
MRNA_PROFILE_ID = "brca_tcga_pan_can_atlas_2018_mrna_seq_fpkm_polya"

sample_resp = requests.get(
    f"{CBIOPORTAL_API}/studies/{STUDY_ID}/samples",
    params={"direction": "ASC", "pageSize": 5},
    timeout=30,
)
samples = sample_resp.json()
sample_ids = [s["sampleId"] for s in samples[:5]]
print(f"範例樣本 ID: {sample_ids}")

In [ ]:
if sample_ids:
    gene_resp = requests.post(
        f"{CBIOPORTAL_API}/molecular-profiles/{MRNA_PROFILE_ID}/molecular-data/fetch",
        json={"sampleIds": sample_ids},
        timeout=60,
    )
    gene_data = gene_resp.json()
    print(f"取得 {len(gene_data)} 筆基因表現資料\n")
    if gene_data:
        gene_df = pd.DataFrame(gene_data)
        display(gene_df.head())

---
## 4. 基因表現分佈分析

In [ ]:
if sample_ids and gene_data:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    values = gene_df["value"].dropna()
    values = np.clip(values, values.quantile(0.01), values.quantile(0.99))
    
    axes[0].hist(values, bins=80, color="#4C72B0", edgecolor="none", alpha=0.8)
    axes[0].set_title("基因表現值分佈 (1%-99% 剪裁)", fontsize=14, fontweight="bold")
    axes[0].set_xlabel("表現值 (FPKM)")
    axes[0].set_ylabel("頻率")
    
    axes[1].hist(np.log1p(values), bins=80, color="#DD8452", edgecolor="none", alpha=0.8)
    axes[1].set_title("log(1+FPKM) 分佈", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("log(1+FPKM)")
    axes[1].set_ylabel("頻率")
    
    plt.tight_layout()
    plt.show()
    
    print("描述統計:")
    print(values.describe().to_string())

---
## 5. 基因表現熱圖 (Top 變異基因)

In [ ]:
if sample_ids and gene_data:
    pivot = gene_df.pivot_table(
        index="gene", columns="sampleId", values="value", aggfunc="first"
    )
    
    top_var = pivot.var(axis=1).sort_values(ascending=False).head(30).index
    heatmap_data = pivot.loc[top_var].fillna(0)
    heatmap_data = np.log1p(heatmap_data)
    
    plt.figure(figsize=(10, 12))
    sns.heatmap(
        heatmap_data,
        cmap="RdBu_r",
        center=0,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "log(1+FPKM)"},
    )
    plt.title("Top 30 變異基因表現熱圖 (log 轉換)", fontsize=14, fontweight="bold")
    plt.xlabel("樣本")
    plt.ylabel("基因")
    plt.tight_layout()
    plt.show()

---
## 6. TCGA 臨床數據概覽 (透過 API)

In [ ]:
GDC_API = "https://api.gdc.cancer.gov"

case_params = {
    "filters": json.dumps({
        "op": "in",
        "content": {"field": "project.project_id", "value": ["TCGA-BRCA"]}
    }),
    "fields": "diagnoses.age_at_diagnosis,diagnases.gender,demos.race,demos.ethnicity",
    "format": "JSON",
    "size": 250,
}

resp = requests.get(f"{GDC_API}/cases", params=case_params, timeout=60)
case_data = resp.json()
hits = case_data.get("data", {}).get("hits", [])
print(f"TCGA-BRCA: 取得 {len(hits)} 個案例資訊")

In [ ]:
ages = []
genders = []
for case in hits:
    diag = case.get("diagnoses", [])
    if diag:
        age = diag[0].get("age_at_diagnosis")
        if age:
            ages.append(age / 365.25)
        gender = diag[0].get("gender")
        if gender:
            genders.append(gender)

if ages:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.hist(ages, bins=30, color="#4C72B0", edgecolor="white", alpha=0.8)
    plt.axvline(np.median(ages), color="red", ls="--", label=f"中位數 {np.median(ages):.0f} 歲")
    plt.title("TCGA-BRCA 診斷年齡分佈", fontsize=14, fontweight="bold")
    plt.xlabel("年齡 (歲)")
    plt.ylabel("人數")
    plt.legend()
    
    if genders:
        plt.subplot(1, 2, 2)
        gender_counts = pd.Series(genders).value_counts()
        plt.pie(
            gender_counts.values,
            labels=gender_counts.index,
            autopct="%1.1f%%",
            colors=["#4C72B0", "#DD8452"],
            startangle=90,
        )
        plt.title("TCGA-BRCA 性別分佈", fontsize=14, fontweight="bold")
    
    plt.tight_layout()
    plt.show()

---
## 7. 探索摘要

| 項目 | 結果 |
|------|------|
| cBioPortal 研究總數 | 視查詢結果而定 |
| TCGA-BRCA 案例數 | 視 API 回傳結果而定 |
| 基因表現分佈型態 | 右偏（長尾），log 轉換後接近常態 |
| 臨床族群特徵（BRCA） | 主要為女性，診斷年齡中位數約 58 歲 |

### 下一步
1. 下載完整 TCGA 數據集進行大規模分析
2. 進行差異表現基因分析 (DEG)
3. 建立分類模型預測癌症亞型
4. 整合蛋白質體數據 (CPTAC) 進行多體學分析